# Конджойнт-эксперимент

Тут я генерирую пары профилей мигрантов для конджойнт-эксперимента, проверяю баланс атрибутов, делаю блочную рандомизацию и симулирую мощность. Всё по шагам.

In [1]:
import random
import numpy as np
import pandas as pd
from itertools import product
from collections import defaultdict, Counter
from scipy.special import expit, logit
from patsy import dmatrix
import statsmodels.formula.api as smf
from linearmodels import OLS

random.seed(42)
np.random.seed(42)

## Симуляция мощности

Начну с самого базового и подберу выборку через симуляцию мощности. В G-Power я уже считала через effect size d для основных эффектов — при средних эффектах хватало 250 респондентов при 15 задачах. Но хочу посмотреть, как число уровней атрибутов влияет на мощность при совсем маленьком AMCE = 0.05 (это и есть нижняя граница того, что мне интересно обнаружить).

In [2]:
# параметры симуляции
n_simulations = 2000
alpha = 0.05
levels  = [7, 4, 2, 2, 3, 4, 2]  # число уровней по каждому атрибуту

In [3]:
def compute_power(n_respondents, n_tasks, amce, levels, n_simulations, alpha=0.05, seed=42):
    ### Симулирует эксперимент n_simulations раз и считает долю случаев,
    ### когда AMCE оказался статистически значимым. Это и есть мощность.
    rng = np.random.default_rng(seed)
    n_significant = 0

    for _ in range(n_simulations):
        choices = []
        dummies = []

        for _ in range(n_respondents * n_tasks):
            # случайно генерируем два профиля
            profile_a = [rng.integers(0, L) for L in levels]
            profile_b = [rng.integers(0, L) for L in levels]

            # вероятность выбрать A зависит от того, у кого нужный уровень первого атрибута
            prob = 0.5 + amce * (int(profile_a[0] == 1) - int(profile_b[0] == 1))
            prob = np.clip(prob, 0, 1)

            choices.append(int(rng.random() < prob))
            dummies.append(int(profile_a[0] == 1) - int(profile_b[0] == 1))

        y = np.array(choices, dtype=float)
        x = np.array(dummies,  dtype=float)

        denom = ((x - x.mean()) ** 2).sum()
        if denom == 0:
            continue

        # оцениваем AMCE через OLS вручную
        beta = ((x - x.mean()) * (y - y.mean())).sum() / denom
        resid = y - (beta * x + y.mean() - beta * x.mean())
        se = np.sqrt((resid ** 2).sum() / (len(y) - 2) / denom)

        if se > 0 and abs(beta / se) > 1.96:
            n_significant += 1

    return n_significant / n_simulations


print(f"{'N':>5}  {'tasks':>6}  {'power':>8}")
for n in [250, 300, 365, 400, 500]:
    power = compute_power(n, 15, 0.05, levels, n_simulations)
    print(f"{n:>5}  {'15':>6}  {power:.3f}")

    N   tasks     power
  250      15  0.860
  300      15  0.919
  365      15  0.962
  400      15  0.972
  500      15  0.986


## Атрибуты и параметры эксперимента

Определяю атрибуты мигрантов и их уровни. Выбор уровней основан на литературе по конджойнтам про иммиграционные установки (Hainmueller & Hopkins 2015). Параметры выборки: 365 респондентов, 15 заданий на каждого.

In [4]:
attributes = {
    'country':    ['Узбекистан', 'Индия', 'Пакистан', 'Венгрия', 'Румыния', 'Беларусь', 'Украина'],
    'motivation': ['Поиск работы', 'Воссоединение с супругом(ой)', 'Политическая нестабильность', 'Учёба'],
    'employer': ['Государственная организация', 'Небольшая частная компания'],
    'gender': ['Мужчина', 'Женщина'],
    'age': [21, 48, 62],
    'occupation': ['Врач', 'Программист', 'Строитель', 'Сфера услуг (общепит)'],
    'language':   ['Говорит свободно', 'Говорит плохо'],
}

n_respondents = 365
n_tasks = 15

# сколько параметров b надо оценить в модели (для каждого атрибута — число уровней минус 1)
n_params = sum(len(v) - 1 for v in attributes.values())
print(f'Параметров в модели: {n_params}')
print(f'Всего оценок от респондентов: {n_respondents * n_tasks}')

Параметров в модели: 17
Всего оценок от респондентов: 5475


## Генерация реалистичных профилей

Сначала генерирую все возможные комбинации атрибутов, потом убираю нереалистичные. Нереалистичные — это те, где комбинация атрибутов не встречается в реальной жизни.

In [5]:
def is_realistic(profile):
    # врач в 21 год - не бывает, нужно минимум лет 8 после школы
    if profile['age'] == 21 and profile['occupation'] == 'Врач':
        return False
    # учёба как мотив миграции в 48 и 62 года — крайне редко
    if profile['motivation'] == 'Учёба' and profile['age'] in [48, 62]:
        return False
    return True


all_profiles = [
    dict(zip(attributes.keys(), combo))
    for combo in product(*attributes.values())
]

realistic = [p for p in all_profiles if is_realistic(p)]
profiles  = pd.DataFrame(realistic).reset_index(drop=True)

print(f'Всего профилей:{len(all_profiles)}')
print(f'Нереалистичных: {len(all_profiles) - len(realistic)}')
print(f'Реалистичных: {len(profiles)}')
print(f'Возможных пар: {len(profiles) * (len(profiles) - 1) // 2:,}')

Всего профилей:2688
Нереалистичных: 672
Реалистичных: 2016
Возможных пар: 2,031,120


## Сколько пар нужно? TVD-симуляция

TVD (total variation distance) - это среднее отклонение вероятностей уровней в выборке пар от их вероятностей в генеральной совокупности. Чем меньше TVD, тем лучше выборка аппроксимирует генеральную совокупность. Эмпирически предположим, что должен быть TVD < 0.05.

Сначала считаю истинные вероятности каждого уровня в генеральной совокупности реалистичных профилей.

In [36]:
# истинные вероятности уровней в генеральной совокупности
true_probs = {}
for attr in attributes:
    counts = profiles[attr].value_counts()
    true_probs[attr] = (counts / counts.sum()).to_dict()

print('Вероятности уровней в генеральной совокупности:')
for attr, probs in true_probs.items():
    print(f'\n{attr}')
    for lvl, p in sorted(probs.items(), key=lambda x: -x[1]):
        print(f'  {str(lvl):} {p:.4f}')

Вероятности уровней в генеральной совокупности:

country
  Узбекистан 0.1429
  Индия 0.1429
  Пакистан 0.1429
  Венгрия 0.1429
  Румыния 0.1429
  Беларусь 0.1429
  Украина 0.1429

motivation
  Поиск работы 0.3056
  Воссоединение с супругом(ой) 0.3056
  Политическая нестабильность 0.3056
  Учёба 0.0833

employer
  Государственная организация 0.5000
  Небольшая частная компания 0.5000

gender
  Мужчина 0.5000
  Женщина 0.5000

age
  21 0.3333
  48 0.3333
  62 0.3333

occupation
  Программист 0.2778
  Строитель 0.2778
  Сфера услуг (общепит) 0.2778
  Врач 0.1667

language
  Говорит свободно 0.5000
  Говорит плохо 0.5000


In [38]:
def simulate_tvd(n_pairs, n_sim=100):
    idx = profiles.index.values
    tvd_list = []

    for seed in range(n_sim):
        rng  = np.random.default_rng(seed)
        pairs_i = rng.choice(idx, size=n_pairs)
        pairs_j = rng.choice(idx, size=n_pairs)

        tvd_attr = []
        for attr in attributes:
            all_vals = np.concatenate([
                profiles[attr].iloc[pairs_i].values,
                profiles[attr].iloc[pairs_j].values
            ])
            unique, cnts = np.unique(all_vals, return_counts=True)
            est = dict(zip(unique, cnts / cnts.sum()))
            tvd_attr.append(
                0.5 * sum(abs(true_probs[attr].get(l, 0) - est.get(l, 0)) for l in attributes[attr])
            )
        tvd_list.append(np.mean(tvd_attr))

    return np.mean(tvd_list), np.std(tvd_list)


print(f"{'Пар':>6}  {'Среднее TVD':>12}  {'Std':>8}")
for n in [30, 50, 75, 100, 150, 200, 300, 400, 500, 750, 1000]:
    mean_tvd, std_tvd = simulate_tvd(n)
    print(f'{n:>6}  {mean_tvd:>12.4f}  {std_tvd:>8.4f}')

   Пар   Среднее TVD       Std
    30        0.0766    0.0157
    50        0.0601    0.0115
    75        0.0504    0.0092
   100        0.0428    0.0085
   150        0.0349    0.0080
   200        0.0305    0.0071
   300        0.0247    0.0058
   400        0.0208    0.0046
   500        0.0186    0.0037
   750        0.0154    0.0027
  1000        0.0132    0.0025


In [8]:
# ищем минимальное число пар при котором TVD < 0.05
def find_min_pairs(candidate_ns, threshold=0.05, n_sim=100):
    results = []
    for n in candidate_ns:
        mean_tvd, std_tvd = simulate_tvd(n, n_sim=n_sim)
        results.append({'n_pairs': n, 'mean_tvd': mean_tvd, 'std_tvd': std_tvd,
                        'score': mean_tvd + std_tvd})
    df = pd.DataFrame(results)
    eligible = df[df['score'] <= threshold]
    min_n = int(eligible['n_pairs'].iloc[0]) if len(eligible) > 0 else None
    return df, min_n


candidate_ns = [30, 50, 75, 90, 100, 115, 150, 200, 300, 400, 500]
results_df, min_pairs = find_min_pairs(candidate_ns, threshold=0.05)

print(results_df.to_string(index=False))
print(f'Минимальное число пар при TVD < 0.05: {min_pairs}')

 n_pairs  mean_tvd  std_tvd    score
      30  0.076557 0.015748 0.092305
      50  0.060054 0.011478 0.071531
      75  0.050351 0.009226 0.059577
      90  0.044833 0.008750 0.053583
     100  0.042783 0.008501 0.051284
     115  0.039587 0.008509 0.048097
     150  0.034854 0.007999 0.042853
     200  0.030488 0.007129 0.037618
     300  0.024655 0.005812 0.030468
     400  0.020817 0.004636 0.025453
     500  0.018613 0.003696 0.022309
Минимальное число пар при TVD < 0.05: 115


## Генерация 450 пар с аппроксимацией генеральной совокупности

Алгоритм работает так: на каждом шаге выбираем два профиля, у которых уровни атрибутов пока недопредставлены в уже отобранных парах (deficit). Это обеспечивает, что итоговое распределение уровней в 450 парах будет близко к генеральной совокупности.

Профили должны отличаться хотя бы по двум атрибутам -
иначе пара слишком похожая и респондент не сможет нормально выбрать.

In [10]:
def deficit(i, total_counts):
    ### насколько уровни профиля i недопредставлены в уже отобранных парах. Чем больше — тем нужнее этот профиль.
    score = 0
    for attr in attributes:
        lvl      = profiles.at[i, attr]
        n_seen   = total_counts[(attr, lvl)]
        n_total  = sum(total_counts[(attr, l)] for l in attributes[attr])
        observed = n_seen / n_total if n_total > 0 else 0
        score   += true_probs[attr].get(lvl, 0) - observed
    return score


def sample_pairs(n_pairs, seed=42):
    random.seed(seed)
    idx   = list(profiles.index)
    selected     = set()
    total_counts = defaultdict(int)

    for _ in range(n_pairs * 500):
        if len(selected) >= n_pairs:
            break

        # профиль 1: из случайного пула берём наименее представленный
        pool_1  = random.sample(idx, min(300, len(idx)))
        profile_1 = max(pool_1, key=lambda i: deficit(i, total_counts))

        # профиль 2: то же самое, но не повторять уже отобранные пары
        pool_2 = random.sample([i for i in idx if i != profile_1], min(150, len(idx) - 1))
        profile_2 = None
        best = float('-inf')

        for candidate in pool_2:
            pair = (min(profile_1, candidate), max(profile_1, candidate))
            if pair in selected:
                continue
            # профили должны отличаться хотя бы по двум атрибутам
            n_diff = sum(profiles.at[profile_1, a] != profiles.at[candidate, a] for a in attributes)
            if n_diff < 2:
                continue
            score = deficit(candidate, total_counts)
            if score > best:
                best, profile_2 = score, candidate

        if profile_2 is None:
            continue

        pair = (min(profile_1, profile_2), max(profile_1, profile_2))
        selected.add(pair)

        for attr in attributes:
            total_counts[(attr, profiles.at[profile_1, attr])] += 1
            total_counts[(attr, profiles.at[profile_2, attr])] += 1

    return list(selected)


selected_pairs = sample_pairs(n_pairs=450)
print(len(selected_pairs))

450


## Проверка баланса

Три проверки: TVD (насколько близко к генеральной совокупности), CV (коэффициент вариации уровней внутри атрибута), chi2 (независимость атрибутов между собой).

In [39]:
def check_tvd(selected_pairs):
    counts = defaultdict(int)
    for i, j in selected_pairs:
        for attr in attributes:
            counts[(attr, profiles.at[i, attr])] += 1
            counts[(attr, profiles.at[j, attr])] += 1

    tvd_by_attr = {}
    for attr in attributes:
        s   = sum(counts[(attr, l)] for l in attributes[attr])
        est = {l: counts[(attr, l)] / s for l in attributes[attr]}
        tvd_by_attr[attr] = 0.5 * sum(
            abs(true_probs[attr].get(l, 0) - est.get(l, 0)) for l in attributes[attr]
        )
    return tvd_by_attr, np.mean(list(tvd_by_attr.values()))


tvd_by_attr, mean_tvd = check_tvd(selected_pairs)
print(f'Среднее TVD: {mean_tvd:.4f}  (цель < 0.05)\n')
for attr, tvd in tvd_by_attr.items():
    print(f'{attr:} {tvd:.4f}')

Среднее TVD: 0.0006  (цель < 0.05)

country 0.0019
motivation 0.0000
employer 0.0000
gender 0.0011
age 0.0000
occupation 0.0000
language 0.0011


In [40]:
# CV — коэффициент вариации: насколько равномерно уровни встречаются в парах
# идеально когда CV < 15%, то есть уровни встречаются примерно одинаково часто

rows_for_df = []
for i, j in selected_pairs:
    for idx in [i, j]:
        rows_for_df.append({a: profiles.at[idx, a] for a in attributes})
attr_df = pd.DataFrame(rows_for_df)

print('Баланс уровней по атрибутам:')
for attr in attributes:
    counts_attr = attr_df[attr].value_counts()
    cv = counts_attr.std() / counts_attr.mean() * 100
    print(f'\n{attr}  (CV={cv:.1f}%)')
    for level, count in counts_attr.items():
        print(f'{str(level):} {count}')

Баланс уровней по атрибутам:

country  (CV=0.4%)
Беларусь 129
Узбекистан 129
Индия 129
Румыния 129
Венгрия 128
Украина 128
Пакистан 128

motivation  (CV=44.4%)
Воссоединение с супругом(ой) 275
Политическая нестабильность 275
Поиск работы 275
Учёба 75

employer  (CV=0.0%)
Государственная организация 450
Небольшая частная компания 450

gender  (CV=0.3%)
Мужчина 451
Женщина 449

age  (CV=0.0%)
62 300
48 300
21 300

occupation  (CV=22.2%)
Программист 250
Сфера услуг (общепит) 250
Строитель 250
Врач 150

language  (CV=0.3%)
Говорит свободно 451
Говорит плохо 449


In [42]:
# chi2 — проверяем независимость атрибутов между собой
# большой chi2 означает что атрибуты коррелируют — это плохо для идентификации модели

pairs_to_check = [
    ('country', 'language'),
    ('country', 'occupation'),
    ('country', 'motivation'),
    ('motivation', 'occupation'),
    ('motivation', 'age'),
    ('occupation', 'language'),
    ('age', 'occupation'),
]

for a1, a2 in pairs_to_check:
    ct   = pd.crosstab(attr_df[a1], attr_df[a2])
    expected = np.outer(ct.sum(axis=1), ct.sum(axis=0)) / ct.values.sum()
    chi2  = ((ct.values - expected) ** 2 / expected).sum()
    print(f'{a1} * {a2}   chi2 = {chi2:.1f}')

country * language   chi2 = 4.4
country * occupation   chi2 = 26.8
country * motivation   chi2 = 34.5
motivation * occupation   chi2 = 36.3
motivation * age   chi2 = 173.5
occupation * language   chi2 = 6.6
age * occupation   chi2 = 106.3


In [14]:
rows = []
for pair_id, (i, j) in enumerate(selected_pairs, 1):
    row = {'pair_id': pair_id}
    for attr in attributes:
        row[f'A_{attr}'] = profiles.at[i, attr]
        row[f'B_{attr}'] = profiles.at[j, attr]
    rows.append(row)

pairs_df = pd.DataFrame(rows)
pairs_df.to_excel('conjoint_pairs_450.xlsx', index=False)

## Блочная рандомизация

Раскладываю 450 пар на 30 блоков по 15 штук и так, чтобы каждый блок тоже был сбалансирован по атрибутам — иначе разные группы респондентов будут видеть систематически разные профили.

Алгоритм: сначала сортирую пары по редкости уровней (редкие уровни — в первую очередь), потом с помощью жадного алгоритма кладу каждую пару в тот блок, где она создаёт наименьший дисбаланс.

In [15]:
attrs = [c[2:] for c in pairs_df.columns if c.startswith('A_')]
n_blocks  = len(pairs_df) // n_tasks  # = 30

# считаем частоты уровней по всем парам
all_counts = []
for i in range(len(pairs_df)):
    row    = pairs_df.iloc[i]
    counts = {a: Counter([row[f'A_{a}'], row[f'B_{a}']]) for a in attrs}
    all_counts.append(counts)

global_counts = {a: Counter() for a in attrs}
for counts in all_counts:
    for a in attrs:
        global_counts[a] += counts[a]

#сколько раз уровень должен встречаться в одном блоке
target_per_block = {
    a: {lvl: cnt / n_blocks for lvl, cnt in global_counts[a].items()}
    for a in attrs
}

print(f'Блоков: {n_blocks}, пар в блоке: {n_tasks}')

Блоков: 30, пар в блоке: 15


In [16]:
def imbalance(block_counts):
    return sum(
        (block_counts[a].get(lvl, 0) - t) ** 2
        for a in attrs
        for lvl, t in target_per_block[a].items()
    )


# редкие уровни идут первыми — чтобы они попали в блоки равномерно
rarity = [
    sum(1 / global_counts[a][pairs_df.iloc[i][f'A_{a}']] +
        1 / global_counts[a][pairs_df.iloc[i][f'B_{a}']]
        for a in attrs)
    for i in range(len(pairs_df))
]
order = sorted(range(len(pairs_df)), key=lambda i: rarity[i], reverse=True)

# жадная раскладка
blocks = {b: [] for b in range(n_blocks)}
block_counts = {b: {a: Counter() for a in attrs} for b in range(n_blocks)}

for i in order:
    best_block = None
    best_delta = float('inf')

    for b in range(n_blocks):
        if len(blocks[b]) >= n_tasks:
            continue
        temp  = {a: block_counts[b][a] + all_counts[i][a] for a in attrs}
        delta = imbalance(temp) - imbalance(block_counts[b])
        if delta < best_delta:
            best_delta = delta
            best_block = b

    blocks[best_block].append(i)
    for a in attrs:
        block_counts[best_block][a].update(all_counts[i][a])

print('Пар в каждом блоке:', [len(blocks[b]) for b in range(n_blocks)])

Пар в каждом блоке: [15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15]


In [17]:
# распределяем 365 респондентов по блокам
# дополнительных респондентов отдаём самым несбалансированным блокам
base      = n_respondents // n_blocks
remainder = n_respondents % n_blocks

worst_blocks = sorted(range(n_blocks), key=lambda b: imbalance(block_counts[b]), reverse=True)
extra   = set(worst_blocks[:remainder])

n_resp = {b: base + (1 if b in extra else 0) for b in range(n_blocks)}

print(f'Базово на блок: {base}, с надбавкой: {base + 1}')
print(f'Блоков с надбавкой: {len(extra)}')

Базово на блок: 12, с надбавкой: 13
Блоков с надбавкой: 5


In [34]:
result_rows = []
resp_id = 0

for b in range(n_blocks):
    block_df = pairs_df.iloc[blocks[b]].copy().reset_index(drop=True)
    for _ in range(n_resp[b]):
        tmp = block_df.copy()
        tmp['respondent_id'] = resp_id
        tmp['block_id']      = b
        result_rows.append(tmp)
        resp_id += 1

survey = pd.concat(result_rows, ignore_index=True)
survey.to_excel('survey_blocks_450.xlsx', index=False)

print(f'{n_blocks} блоков * {n_tasks} пар * ~{base} респондентов = {len(survey)} строк')

30 блоков * 15 пар * ~12 респондентов = 5475 строк


## Проверка баланса по блокам

In [19]:
for a in attrs:
    lvls  = sorted(global_counts[a].keys(), key=str)
    table = []
    for b in range(n_blocks):
        row = {'блок': b}
        for lvl in lvls:
            row[str(lvl)] = block_counts[b][a].get(lvl, 0)
        table.append(row)
    df_bal = pd.DataFrame(table).set_index('блок')
    print(f'\n{a}')
    print(df_bal.to_string())


country
      Беларусь  Венгрия  Индия  Пакистан  Румыния  Узбекистан  Украина
блок                                                                  
0            6        5      3         4        5           4        3
1            3        6      3         4        4           5        5
2            4        5      4         3        3           4        7
3            5        5      4         5        3           3        5
4            3        4      5         5        5           4        4
5            4        4      4         5        4           5        4
6            6        4      5         4        4           3        4
7            4        6      4         6        4           3        3
8            5        4      4         4        4           4        5
9            6        4      4         6        2           4        4
10           4        4      4         5        5           5        3
11           3        5      5         4        5           5       

## Симуляция эксперимента и оценка AMCE

Тут задаю гипотетические AMCE (средние маргинальные эффекты компонентов), потом симулирую эксперимент и смотрю, восстанавливает ли модель эти эффекты.


In [20]:
target_amce = {
    'country_Индия': -0.05,
    'country_Пакистан': -0.04,
    'country_Венгрия': 0.07,
    'country_Румыния': 0.06,
    'country_Беларусь': 0.10,
    'country_Украина': 0.09,
    'motivation_Воссоединение с супругом(ой)':  0.05,
    'motivation_Политическая нестабильность':   0.08,
    'motivation_Учёба': 0.06,
    'employer_Небольшая частная компания':  -0.06,
    'gender_Женщина': 0.04,
    'age_48': -0.05,
    'age_62': -0.12,
    'occupation_Программист': -0.06,
    'occupation_Строитель': -0.12,
    'occupation_Сфера услуг (общепит)': -0.14,
    'language_Говорит плохо': -0.15,
}

# переводим AMCE в коэффициенты логит-модели
coefficients = {k: logit(0.5 + v) for k, v in target_amce.items()}

In [21]:
def compute_utility(prefix, row):
    return sum(coefficients.get(f'{attr}_{row[f"{prefix}_{attr}"]}', 0.0) for attr in attrs)


def simulate_experiment(pairs, n_respondents, n_tasks, seed=42):
    np.random.seed(seed)
    rows = []

    for respondent_id in range(n_respondents):
        block_id = respondent_id % (len(pairs) // n_tasks)
        for _, pair in pairs.sample(n=n_tasks, replace=False).iterrows():
            p_a    = expit(compute_utility('A', pair) - compute_utility('B', pair))
            # исправленная формула: выбор стохастический, не детерминированный
            choice = int(p_a > 0.5)
            for prefix, c in [('A', choice), ('B', 1 - choice)]:
                rows.append(
                    {a: pair[f'{prefix}_{a}'] for a in attrs} |
                    {'choice': c, 'respondent_id': respondent_id,
                     'block_id': block_id, 'age': str(pair[f'{prefix}_age'])}
                )
    return pd.DataFrame(rows)

In [22]:
experiment = simulate_experiment(pairs_df, n_respondents, n_tasks)

formula = (
    'choice ~ '
    'C(country,Treatment("Узбекистан")) + '
    'C(motivation, Treatment("Поиск работы")) + '
    'C(employer, Treatment("Государственная организация")) + '
    'C(gender, Treatment("Мужчина")) + '
    'C(age, Treatment("21")) + '
    'C(occupation, Treatment("Врач")) + '
    'C(language, Treatment("Говорит свободно"))'
)

# двойная кластеризация: по респонденту и по блоку
model = OLS.from_formula(formula, data=experiment).fit(
    cov_type='clustered',
    clusters=experiment[['respondent_id', 'block_id']]
)

print(model.summary)

                            OLS Estimation Summary                            
Dep. Variable:                 choice   R-squared:                      0.2499
Estimator:                        OLS   Adj. R-squared:                 0.2487
No. Observations:               10950   F-statistic:                 1.853e+04
Date:                Fri, Apr 10 2026   P-value (F-stat)                0.0000
Time:                        15:08:28   Distribution:                 chi2(17)
Cov. Estimator:             clustered                                         
                                                                              
                                                                  Parameter Estimates                                                                  
                                                                                     Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
----------------------------------------------------------------

In [23]:
# проверка матрицы регрессоров: нет ли мультиколлинеарности
X = dmatrix(formula.replace('choice ~ ', ''), data=experiment, return_type='dataframe')
X = X.drop('Intercept', axis=1)

rank = np.linalg.matrix_rank(X.values)
singular_values = np.linalg.svd(X.values, compute_uv=False)
condition_number = singular_values.max() / singular_values.min()

zero_var = X.columns[X.var() == 0].tolist()

print(f'Столбцов: {X.shape[1]}, ранг: {rank}, полный ранг: {rank == X.shape[1]}')
print(f'Condition number: {condition_number:.1f}  {"(ОК)" if condition_number < 1000 else "не ок(("}' )
print(f'Нулевая дисперсия: {zero_var if zero_var else "нет"}')

Столбцов: 17, ранг: 17, полный ранг: True
Condition number: 8.8  (ОК)
Нулевая дисперсия: нет




# Часть 2. Генерация 100 пар (полная рандомизация)

Вторая часть - вариант с 100 парами. Логика отбора та же: с аппроксимацией генеральной совокупности. Отличие от 450 пар в том, что здесь не делаем сложную блочную рандомизацию с балансировкой.

## Отбор 100 пар

In [26]:
# функции deficit, sample_pairs, check_tvd уже определены выше
selected_pairs_100 = sample_pairs(n_pairs=100, seed=42)
print(len(selected_pairs_100))

100


## Проверка баланса

In [43]:
tvd_by_attr_100, mean_tvd_100 = check_tvd(selected_pairs_100)
print(f'Среднее TVD: {mean_tvd_100:.4f}  (цель < 0.05)\n')
for attr, tvd in tvd_by_attr_100.items():
    print(f'{attr:} {tvd:.4f}')

Среднее TVD: 0.0034  (цель < 0.05)

country 0.0086
motivation 0.0044
employer 0.0000
gender 0.0000
age 0.0033
occupation 0.0072
language 0.0000


In [44]:
# CV по атрибутам
rows_for_check_100 = []
for i, j in selected_pairs_100:
    for idx in [i, j]:
        rows_for_check_100.append({a: profiles.at[idx, a] for a in attributes})
attr_df_100 = pd.DataFrame(rows_for_check_100)

print('Баланс уровней по атрибутам:')
for attr in attributes:
    counts_attr = attr_df_100[attr].value_counts()
    cv = counts_attr.std() / counts_attr.mean() * 100
    print(f'\n{attr}  (CV={cv:.1f}%)')
    for level, count in counts_attr.items():
        print(f'{str(level)} {count}')

Баланс уровней по атрибутам:

country  (CV=1.9%)
Узбекистан 29
Украина 29
Индия 29
Румыния 29
Венгрия 28
Беларусь 28
Пакистан 28

motivation  (CV=45.3%)
Воссоединение с супругом(ой) 62
Политическая нестабильность 61
Поиск работы 61
Учёба 16

employer  (CV=0.0%)
Государственная организация 100
Небольшая частная компания 100

gender  (CV=0.0%)
Женщина 100
Мужчина 100

age  (CV=0.9%)
62 67
21 67
48 66

occupation  (CV=22.7%)
Сфера услуг (общепит) 57
Программист 55
Строитель 55
Врач 33

language  (CV=0.0%)
Говорит плохо 100
Говорит свободно 100


In [29]:
# chi2
pairs_to_check = [
    ('country', 'language'),
    ('country', 'occupation'),
    ('country', 'motivation'),
    ('motivation', 'occupation'),
    ('motivation', 'age'),
    ('occupation', 'language'),
    ('age', 'occupation'),
]

for a1, a2 in pairs_to_check:
    ct  = pd.crosstab(attr_df_100[a1], attr_df_100[a2])
    expected = np.outer(ct.sum(axis=1), ct.sum(axis=0)) / ct.values.sum()
    chi2  = ((ct.values - expected) ** 2 / expected).sum()
    print(f'{a1} × {a2}   chi2 = {chi2:.1f}')

country × language   chi2 = 13.2
country × occupation   chi2 = 28.9
country × motivation   chi2 = 21.4
motivation × occupation   chi2 = 19.4
motivation × age   chi2 = 41.7
occupation × language   chi2 = 3.1
age × occupation   chi2 = 23.5


## Сохраняем 100 пар и делаем блоки

In [30]:
rows_100 = []
for pair_id, (i, j) in enumerate(selected_pairs_100, 1):
    row = {'pair_id': pair_id}
    for attr in attributes:
        row[f'A_{attr}'] = profiles.at[i, attr]
        row[f'B_{attr}'] = profiles.at[j, attr]
    rows_100.append(row)

pairs_df_100 = pd.DataFrame(rows_100)
pairs_df_100.to_excel('conjoint_pairs_100.xlsx', index=False)

## Симуляция эксперимента для 100 пар

Кластеризация только по респонденту — блоков слишком мало (~6), чтобы кластеризация по блоку давала корректные стандартные ошибки (Cameron & Miller 2015 рекомендуют не меньше 20-50 кластеров).

In [31]:
experiment_100 = simulate_experiment(pairs_df_100, n_respondents, n_tasks)

formula = (
    'choice ~ '
    'C(country, Treatment("Узбекистан")) + '
    'C(motivation, Treatment("Поиск работы")) + '
    'C(employer, Treatment("Государственная организация")) + '
    'C(gender, Treatment("Мужчина")) + '
    'C(age, Treatment("21")) + '
    'C(occupation, Treatment("Врач")) + '
    'C(language, Treatment("Говорит свободно"))'
)

# кластеризация только по респонденту
model_100 = smf.ols(formula, data=experiment_100).fit(
    cov_type='cluster',
    cov_kwds={'groups': experiment_100['respondent_id']}
)

print(model_100.summary())

                            OLS Regression Results                            
Dep. Variable:                 choice   R-squared:                       0.257
Model:                            OLS   Adj. R-squared:                  0.256
Method:                 Least Squares   F-statistic:                     904.3
Date:                Fri, 10 Apr 2026   Prob (F-statistic):          1.21e-285
Time:                        15:09:06   Log-Likelihood:                -6322.5
No. Observations:               10950   AIC:                         1.268e+04
Df Residuals:                   10932   BIC:                         1.281e+04
Df Model:                          17                                         
Covariance Type:              cluster                                         
                                                                                          coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------

In [32]:
# проверка матрицы
X_100 = dmatrix(formula.replace('choice ~ ', ''), data=experiment_100, return_type='dataframe')
X_100 = X_100.drop('Intercept', axis=1)

rank_100 = np.linalg.matrix_rank(X_100.values)
sv_100   = np.linalg.svd(X_100.values, compute_uv=False)
cond_100 = sv_100.max() / sv_100.min()
zero_var_100 = X_100.columns[X_100.var() == 0].tolist()

print(f'Столбцов: {X_100.shape[1]}, ранг: {rank_100}, полный ранг: {rank_100 == X_100.shape[1]}')
print(f'Condition number: {cond_100:.1f}  {"(ОК)" if cond_100 < 1000 else "не ок(("}')
print(f'Нулевая дисперсия: {zero_var_100 if zero_var_100 else "нет"}')

Столбцов: 17, ранг: 17, полный ранг: True
Condition number: 9.1  (ОК)
Нулевая дисперсия: нет
